# <font style="font-family:roboto;color:#455e6c"> Validation of Interatomic Potentials </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 8 March 2026 </font> </br> </br>
Sriram Anand, Prabhath Chilakalapudi, Haitham Gaafer, Marvin Poul, Jan Janssen, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Sarath Menon, Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>
Build a workflow to <code>iterate</code> over different (hydrostatic) strain states and get energy of the system.
To break it down, we need to
<ol>
    <li>Construct a set of structures that span different volumes
        <ul>
            <li>Construct the initial crystal</li>
            <li>Create an array of strains</li>
            <li>Apply the strains</li>
        </ul>
        <b>Simpler to change the volume by altering the lattice parameter</b>
    </li>
    Hint: Use the <code>IterateToDataframe</code> node
    <li>Calculate the energy at each volume
        <ul>
            <li>Loop over each strained structure</li>
            <li>Run a static single-point energy calculation, so that the volume isn't changed</li>
            <li>Collect the resulting energy for each structure</li>
        </ul>
    </li>
    Which node should we use?  <code>SinglePointStatic</code> or <code>Relax</code> ?
    <li>Gather the results into a usable format
        <ul>
            <li>Extract the volume from each structure</li>
            <li>Pair each volume with its corresponding energy to form (V, E) data points</li>
        </ul>
    </li>
    <li>Fit an equation of state (EOS) to the (V, E) data
        <ul>
            <li>Choose an EOS model (e.g., Birch-Murnaghan)</li>
            <li>Fit the model to extract equilibrium volume V&#8320;, cohesive energy E&#8320;, and bulk modulus B&#8320;</li>
        </ul>
    </li>
    <li>Visualize the results
        <ul>
            <li>Plot the raw (V, E) data points</li>
            <li>Overlay the fitted EOS curve</li>
            <li>Annotate the plot with the extracted quantities</li>
        </ul>
    </li>
</ol>
</div>

In [1]:
from core import Workflow, as_function_node
from core.gui import PyironFlow, GUILayout
from pyiron_nodes.atomistic.calculator.optimize import GenericOptimizerSettings
from pyiron_nodes.atomistic.property.bulk import FitBirchMurnaghanEOS, PlotEVCurve
from pyiron_nodes.atomistic.engine.grace import GRACE
from pyiron_nodes.atomistic.engine.ace import Ace
from pyiron_nodes.atomistic.structure.build import Bulk


from pyiron_nodes.basic.math import Linspace
from pyiron_nodes.basic.loop import IterToDataFrame

from pyiron_nodes.atomistic.calculator.optimize import SinglePointStatic 
from pyiron_nodes.basic.file import DataframeToList

In [2]:
import os

Several pre-fitted GRACE foundation models are available for use. You can choose from the following options:

- `MP_GRACE_1L_r6_4Nov2024`
- `MP_GRACE_1L_r6_07Nov2024`
- `MP_GRACE_2L_r6_11Nov2024`
- `MP_GRACE_2L_r5_4Nov2024`
- `MP_GRACE_2L_r5_07Nov2024`
- `GRACE_2L_OAM_28Jan25`
- `GRACE-1L-OAM_2Feb25`

For further details about the GRACE foundation models, please refer to the [Grace documentation](https://gracemaker.readthedocs.io/en/latest/gracemaker/foundation/).

Note that you only need to provide the model string as an argument—the associated potential files will be downloaded at runtime.


# Single Point Calculation

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Task</b></p>
Build a workflow to <code>iterate</code> over different (hydrostatic) strain states and get energy of the system.
To break it down, we need to
<ol>
    <li>Construct a simple workflow to evaluate the total energy and volume of a given structure
        <ul>
            <li>Construct the initial crystal with a guess-timate lattice parameter: <code>bulk</code></li>
            <li> Define a calculation engine: simplest and fastest - <code>Ace</code></li>  
            <li>Calculate the energy: <code>SinglePointCalculator</code> </li>
        </ul>
    </li>
</ol>
</div>

In [3]:
wf = Workflow("05_01_SinglePoint")

wf.structure = Bulk(name='Ca')
wf.engine = Ace('potentials/Ca_Mg_linear_potential.yace')
wf.SinglePointCalculator = SinglePointStatic()


In [4]:
wf_sol = Workflow("05_01_SinglePoint_solution")

wf_sol.structure = Bulk(name='Ca')
wf_sol.engine = Ace('potentials/Ca_Mg_linear_potential.yace')
wf_sol.SinglePointCalculator = SinglePointStatic(structure = wf_sol.structure,
                                            engine = wf_sol.engine)

In [5]:
from core.gui import PyironFlow

In [6]:
pf = PyironFlow([wf, wf_sol], workflow_path="./pyiron_core_data/workflows/", nodes_path="./pyiron_nodes/")

In [7]:
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">


<p class="title"> <b>Get structures with different volumes over which to iterate </b></p>

<b>Q: Which single parameter can we change to explore different volumes?</b>
<ul>
    <li> Get a list of values to use: <code>Linspace</code></li>
    <li> Functional Programming: Use a node as an input - iterate the node's operation: <code>IterToDataFrame</code> </li>
    
</ul>
</div>

In [12]:
wf_iter = Workflow("05_02_IterToDataframe_intro")

wf_iter.structure = Bulk(name='Ca')
wf_iter.linspace = Linspace()
wf_iter.Iterator = IterToDataFrame()


In [13]:
wf_iter_sol = Workflow("05_02_IterToDataframe_solution")

wf_iter_sol.structure = Bulk(name='Ca')
wf_iter_sol.linspace = Linspace()
wf_iter_sol.Iterator = IterToDataFrame(node = wf_iter_sol.structure,
                                      input_label='a',
                                      values = wf_iter_sol.linspace)

In [14]:
pf_iter = PyironFlow([wf_iter, wf_iter_sol], workflow_path="./pyiron_core_data/workflows/", nodes_path="./pyiron_nodes/")

In [15]:
pf_iter.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">


<p class="title"> <b> Task: Merge the two </b></p>

<ul>
    <li> The <code>IterToDataFrame</code> node only takes a list or array as input for values, what should we do? <i> search for nodes to convert dataframe to list</i></li>
    <li> <b>Note:</b> We need to iterate the single point calculation over the list of structures</li>
    <li> Functional Programming: Use a node as an input - iterate the node's operation: <code>IterToDataFrame</code> </li>
    
</ul>
</div>

In [16]:
workflow_name='05_03_Murnaghan'
potential_path = None
alat_min=None
alat_max=None
Symbol= 'Ca'
max_steps = 10
num_points = 80
force_tolerance =0.01

wf_murn = Workflow(workflow_name)
wf_murn.a_list = Linspace(x_min= alat_min,
                    x_max = alat_max,
                    num_points = num_points, endpoint = True)

wf_murn.structure = Bulk()

wf_murn.structs_df = IterToDataFrame(node=wf_murn.structure,
                                    input_label = 'a',
                                    values = wf_murn.a_list)
wf_murn.engine = Ace()

# wf_murn.opt_settings = GenericOptimizerSettings(max_steps=max_steps, force_tolerance=force_tolerance)

wf_murn.singlepoint = SinglePointStatic(engine = wf_murn.engine, structure = wf_murn.structure)

# wf_murn.DF2List = DataframeToList(wf_murn.structs_df, column = 'structure')

wf_murn.IterateCalculations = IterToDataFrame()

wf_murn.plotting = PlotEVCurve(wf_murn.IterateCalculations)

wf_murn.ExtractData = FitBirchMurnaghanEOS(wf_murn.IterateCalculations)


In [17]:


from typing import Union, Optional
import os
def make_ev_workflows(workflow_name:str,
                      potential_path:Union[str,os.path],
                      alat_min:float,
                      alat_max:float,
                      Symbol:str = 'Ca',
                      max_steps:int = 10,
                        num_points:int = 80,
                      force_tolerance:float=0.01,
                        run:bool=False):

    wf = Workflow(workflow_name)
    wf.a_list = Linspace(x_min= alat_min,
                        x_max = alat_max,
                        num_points = num_points, endpoint = True)
    
    wf.structure = Bulk(name=Symbol)
    
    wf.IterateStructures = IterToDataFrame(node=wf.structure,
                                    input_label = 'a',
                                    values = wf.a_list)
    
    wf.engine = Ace(potential_path)
    
    wf.singlepoint = SinglePointStatic(engine = wf.engine, structure = wf.structure)
    
    wf.DF2List = DataframeToList(wf.IterateStructures, column = 'structure')
    
    wf.IterateCalculations = IterToDataFrame(node = wf.singlepoint, input_label = 'structure', values=wf.DF2List)
    
    wf.plotting = PlotEVCurve(wf.IterateCalculations)
    
    wf.ExtractData = FitBirchMurnaghanEOS(wf.IterateCalculations)

    if run:
        wf.run()

    return wf


In [21]:
workflow_name = '05_03_Murnaghan_solution'
potential_path = "potentials/Ca_Mg_linear_potential.yace"
max_steps = 10,
force_tolerance = 0.01
wf_murn_sol = make_ev_workflows(workflow_name, alat_min=4.0, alat_max=8.0, potential_path=potential_path, max_steps=max_steps, force_tolerance=force_tolerance)

In [22]:
pf_murn = PyironFlow([wf_murn, wf_murn_sol], workflow_path="./pyiron_core_data/workflows/", nodes_path="./pyiron_nodes/")

In [23]:
pf_murn.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"> <b> Task: Use different Engines </b></p>
<ul>
    <li> The <code>Ace</code> node gives an ASE engine to evaluate the energies and forces using an ACE interatomic potential. Try using <code>GRACE</code> engine</li>
    <li> Benchmarking means that we compare a computed property in the fit model against the "Ground truth / reality". Try the <code>QuantumEspresso</code> engine</li>  
</ul>
</div>

<div class="admonition warning" name="html-admonition" style="background: #FFECEC; padding: 10px; border-left: 4px solid #FF4C4C; margin-top: 8px">
<p class="title"> <b> ⚠️ Warning: QuantumEspresso and GRACE are computationally more intensive than ACE</b></p>
<p> Before running with the <code>QuantumEspresso</code> or <code>GRACE</code> engines, <b>please reduce the number of points</b> in your calculation. Using a lot of points can make the workflow run very slow and may crash your Jupyter terminal. </p>
<p> 💡 <span style="background: #D4EDDA; color: #155724; padding: 2px 6px; border-radius: 4px; font-family: monospace;">Set <b>num_points = 7</b> for <code>QuantumEspresso</code> and <b>num_points = 15</b> for <code>GRACE</code> in the <b>linspace</b> a.k.a <b>a_list</b> node</span> </p>
</div>

In [18]:
pf_murn_qe = PyironFlow([wf_murn_sol], workflow_path="./pyiron_core_data/workflows/", nodes_path="./pyiron_nodes/")

In [19]:
pf_murn_qe.gui